<div style="text-align: center; line-height: 0; padding-top: 9px;">
<img src="https://learningjournal.github.io/pub-resources/logos/scholarnest_academy.jpg" alt="ScholarNest Academy" style="width: 1400px">
</div>

Q1. List all bookings made by a person named Darren Smith as the following.
```
member_id | first_name | last_name | address | facility_id | slots
---------------------------------------------------------------------
```

Ensure the following
1. Show the details of all persons named Darren Smith even if they have not made any bookings
2. Sort the result by number of slots (highers first)
3. List the person with no bookings at the top

In [0]:
from pyspark.sql.functions import expr, col

members_df = spark.table("dev.spark_db.members").alias("m")
bookings_df = spark.table("dev.spark_db.bookings").alias("b")

result_df = (
    members_df.join(bookings_df, expr("m.member_id=b.member_id"), "left")
        .where("m.first_name == 'Darren' and m.last_name == 'Smith'")
        .select("m.member_id", "m.first_name", "m.last_name", "m.address", "b.facility_id", "b.slots")
        .orderBy(col("b.slots").desc_nulls_first())
)

result_df.display()

member_id,first_name,last_name,address,facility_id,slots
37,Darren,Smith,"3 Funktown, Denzington, Boston",null,null
1,Darren,Smith,"8 Bloomsbury Close, Boston",2,9
1,Darren,Smith,"8 Bloomsbury Close, Boston",2,6
1,Darren,Smith,"8 Bloomsbury Close, Boston",2,6
1,Darren,Smith,"8 Bloomsbury Close, Boston",2,6
1,Darren,Smith,"8 Bloomsbury Close, Boston",2,6
1,Darren,Smith,"8 Bloomsbury Close, Boston",2,6
1,Darren,Smith,"8 Bloomsbury Close, Boston",2,6
1,Darren,Smith,"8 Bloomsbury Close, Boston",2,6
1,Darren,Smith,"8 Bloomsbury Close, Boston",2,6


Q2. Show me a bookings report for Darren Smith as the following.

```
facility_name | slots | booking_amount | start_time | member_id | member_name | telephone | address
------------------------------------------------------------------------------------------------------
```
The report must meet the following criteria.

1. Show the details of all persons named Darren Smith even if they have not made any bookings
2. Sort the result by number of slots (highers first)
3. List the person with no bookings at the top

In [0]:
from pyspark.sql.functions import expr, col

members_df = (
    spark.table("dev.spark_db.members")
        .where("first_name='Darren' and last_name='Smith'")
)

bookings_df = spark.table("dev.spark_db.bookings")
facilities_df = spark.table("dev.spark_db.facilities")

joined_df = (
    members_df.alias("m")
        .join(bookings_df.alias("b"), expr("m.member_id = b.member_id"), "left")
        .join(facilities_df.alias("f"), expr("b.facility_id=f.facility_id"), "left")
)

results_df = (
    joined_df.selectExpr(
        "f.facility_name", "b.slots",
        "b.slots * f.member_cost as booking_amount",
        "b.start_time", "b.member_id",
        "concat_ws(' ', m.first_name, m.last_name)  as member_name",
        "m.telephone", "m.address"
    ).orderBy(col("slots").desc_nulls_first())    
)

results_df.display()


facility_name,slots,booking_amount,start_time,member_id,member_name,telephone,address
null,null,null,null,null,Darren Smith,(822) 577-3541,"3 Funktown, Denzington, Boston"
Badminton Court,9,0.0,2022-08-28T13:30:00.000Z,1,Darren Smith,555-555-5555,"8 Bloomsbury Close, Boston"
Badminton Court,6,0.0,2022-07-29T12:00:00.000Z,1,Darren Smith,555-555-5555,"8 Bloomsbury Close, Boston"
Badminton Court,6,0.0,2022-08-07T09:00:00.000Z,1,Darren Smith,555-555-5555,"8 Bloomsbury Close, Boston"
Badminton Court,6,0.0,2022-09-09T13:00:00.000Z,1,Darren Smith,555-555-5555,"8 Bloomsbury Close, Boston"
Badminton Court,6,0.0,2022-09-10T09:00:00.000Z,1,Darren Smith,555-555-5555,"8 Bloomsbury Close, Boston"
Badminton Court,6,0.0,2022-09-30T14:00:00.000Z,1,Darren Smith,555-555-5555,"8 Bloomsbury Close, Boston"
Badminton Court,6,0.0,2022-07-09T09:00:00.000Z,1,Darren Smith,555-555-5555,"8 Bloomsbury Close, Boston"
Badminton Court,6,0.0,2022-09-07T14:00:00.000Z,1,Darren Smith,555-555-5555,"8 Bloomsbury Close, Boston"
Badminton Court,6,0.0,2022-07-27T12:00:00.000Z,1,Darren Smith,555-555-5555,"8 Bloomsbury Close, Boston"


Q3. Prepare a facility booking report as the following
```
facility_name | member_cost | gest_cost | start_time | slots
---------------------------------------------------------------
```
Ensure the following
1. All club facilities must be listed in the report
2. Consider only bookings for more than 10 slots

In [0]:
from pyspark.sql.functions import expr

facilities_df = spark.table("dev.spark_db.facilities")
bookings_df = spark.table("dev.spark_db.bookings").filter("slots > 10")

result_df = (
    bookings_df.join(facilities_df, bookings_df.facility_id == facilities_df.facility_id, "right")
             .select(facilities_df.facility_name,
                    facilities_df.member_cost,
                    facilities_df.guest_cost,
                    bookings_df.start_time,
                    bookings_df.slots)
)

result_df.display()


facility_name,member_cost,guest_cost,start_time,slots
Tennis Court 1,5.0,25.0,2022-09-15T08:00:00.000Z,12
Tennis Court 2,5.0,25.0,null,null
Badminton Court,0.0,15.5,null,null
Table Tennis,0.0,5.0,null,null
Massage Room 1,35.0,80.0,null,null
Massage Room 2,35.0,80.0,null,null
Squash Court,3.5,17.5,2022-09-13T10:30:00.000Z,14
Snooker Table,0.0,5.0,null,null
Pool Table,0.0,5.0,null,null


Q4. Prepare a member bookings report as the following
```
booking_id | facility_name | slots | first_name | last_name | address
```
Ensure the following
1. Consider only regular memebrs (not guest) and direct members(not recomended by any other member)
2. Consider only bookings for more than 8 hours
3. Ensure all regular and direct members are listed even if they have no 8 hour bookings
4. Ensure all 8 hour bookings are listed even if they are not made by regular and direct members
5. Sort the report by slots and first name in ascending order

In [0]:
from pyspark.sql.functions import expr

members_df = (
    spark.table("dev.spark_db.members")
        .filter("member_id != 0 and recommended_by is null")
        .alias("m")
)

bookings_df = (
    spark.table("dev.spark_db.bookings")
        .filter("slots > 8")
        .alias("b")
)

facilities_df = spark.table("dev.spark_db.facilities").alias("f")

full_join_df = members_df.join(bookings_df, expr("m.member_id == b.member_id"), "full")

result_df = (
    full_join_df.join(facilities_df, expr("b.facility_id == f.facility_id"), "left")
    .select("b.booking_id","f.facility_name","b.slots","m.first_name","m.last_name","m.address")
    .orderBy(expr("b.slots").asc_nulls_last(), expr("m.first_name").asc_nulls_last())
)

display(result_df)

booking_id,facility_name,slots,first_name,last_name,address
1927,Badminton Court,9,Darren,Smith,"8 Bloomsbury Close, Boston"
1757,Tennis Court 2,9,null,null,null
3563,Tennis Court 1,9,null,null,null
660,Tennis Court 2,9,null,null,null
3836,Tennis Court 2,9,null,null,null
3041,Tennis Court 1,9,null,null,null
530,Tennis Court 1,9,null,null,null
3768,Tennis Court 2,9,null,null,null
2978,Tennis Court 1,12,null,null,null
2888,Squash Court,14,null,null,null


&copy; 2021-2026 <a href="https://www.scholarnest.com/">ScholarNest</a>. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the <a href="https://www.apache.org/">Apache Software Foundation.</a><br/>
Databricks, Databricks Cloud and the Databricks logo are trademarks of the <a href="https://www.databricks.com/">Databricks Inc.</a><br/>
<a href="https://www.scholarnest.com/pages/privacy">Privacy Policy</a> | <a href="https://www.scholarnest.com/pages/terms">Terms of Use</a> | <a href="https://www.scholarnest.com/pages/contact">Contact Us</a>